In [16]:
import pandas as pd
import numpy as np
import networkx as nx
from math import radians, sin, cos, sqrt, atan2
import networkx as nx

In [3]:
route_stops = pd.read_csv("route_stops_full.csv")
route_stops.head()

,PublishedLineName,DirectionRef,Stop Name,stoplat,stoplon,obs
0,B35,0.0,14 AV/36 ST,40.641006,-73.982306,925
1,B35,0.0,14 AV/39 ST,40.639481,-73.984300,1408
2,B35,0.0,14 AV/CHURCH AV,40.641736,-73.981549,1037
3,B35,0.0,39 ST/10 AV,40.644695,-73.993030,784
4,B35,0.0,39 ST/12 AV,40.642054,-73.988660,997


In [5]:
#Build Nodes
nodes = (
    route_stops.groupby("Stop Name", as_index=False)
    .agg(
        stop_lat=("stoplat", "median"),
        stop_lon=("stoplon", "median"),
        total_obs=("obs", "sum")
    )
)

nodes.head()

,Stop Name,stop_lat,stop_lon,total_obs
0,108 ST/53 AV,40.742945,-73.854753,4215
1,108 ST/HORACE HARDING EP,40.737824,-73.851798,2983
2,108 ST/HORACE HARDING EXPWY,40.738070,-73.852257,2833
3,108 ST/MARTENSE AV,40.742284,-73.854417,1034
4,108 ST/OTIS AV,40.741549,-73.854038,2345


In [13]:
def haversine(lat1, lon1, lat2, lon2):
    R = 6371000
    dlat = radians(lat2 - lat1)
    dlon = radians(lon2 - lon1)
    a = sin(dlat / 2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlon / 2)**2
    return 2 * R * atan2(sqrt(a), sqrt(1 - a))

G = nx.DiGraph()

# Add nodes
for _, row in nodes.iterrows():
    G.add_node(
        row["Stop Name"],
        stop_lat=row["stop_lat"],
        stop_lon=row["stop_lon"],
        total_obs=row["total_obs"]
    )

# Add route-direction edges
for (route, direction), group in route_stops.groupby(["PublishedLineName", "DirectionRef"]):
    group = group.drop_duplicates(subset=["Stop Name"]).copy()

    lon_span = group["stoplon"].max() - group["stoplon"].min()
    lat_span = group["stoplat"].max() - group["stoplat"].min()

    if lon_span >= lat_span:
        group = group.sort_values("stoplon", ascending=(direction == 0.0))
    else:
        group = group.sort_values("stoplat", ascending=(direction == 0.0))

    rows = group[["Stop Name", "stoplat", "stoplon", "obs"]].to_dict("records")

    for i in range(len(rows) - 1):
        a = rows[i]
        b = rows[i + 1]

        dist_m = haversine(a["stoplat"], a["stoplon"], b["stoplat"], b["stoplon"])

        if dist_m > 3000:
            continue

        G.add_edge(
            a["Stop Name"],
            b["Stop Name"],
            route=route,
            direction=direction,
            distance_m=dist_m,
            from_obs=a["obs"],
            to_obs=b["obs"],
            weight=dist_m
        )

print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())

Nodes: 406
Edges: 527


In [14]:
edges = []

for u, v, data in G.edges(data=True):
    edges.append({
        "from_stop": u,
        "to_stop": v,
        "route": data["route"],
        "direction": data["direction"],
        "distance_m": data["distance_m"],
        "from_obs": data["from_obs"],
        "to_obs": data["to_obs"],
        "weight": data["weight"]
    })

edges_df = pd.DataFrame(edges)

nodes.to_csv("graph_nodes.csv", index=False)
edges_df.to_csv("graph_edges.csv", index=False)

In [18]:
# Save Graph as GEXF
nx.write_gexf(G, "bus_stop_graph.gexf")